<a href="https://colab.research.google.com/github/sadumina/Deep-Learning-Assignment-Group-ID-5/blob/model%2FDenseNet121/train_densenet121_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetic Retinopathy Classification — DenseNet121 (Transfer Learning)

**Dataset:** APTOS 2019 Blindness Detection, pre-processed version *Diabetic Retinopathy 224x224 Gaussian Filtered* (Kaggle, 3,662 fundus images).
**Task:** 5-class severity grading — No DR, Mild, Moderate, Severe, Proliferative DR.
**Approach:** DenseNet121 pre-trained on ImageNet, trained in two stages:
1. **Feature extraction** — backbone frozen, only a new classification head is trained.
2. **Fine-tuning** — the last 60 backbone layers are unfrozen and trained with a very small learning rate.

**Fair comparison:** the train/validation split is identical to the Classic-CNN model (80/20, `seed=42`).

> Before running: **Runtime → Change runtime type → T4 GPU**

## 1. Environment setup
Mount Google Drive (to save results permanently) and import the libraries.

In [1]:
# Mount Google Drive so checkpoints and plots survive a Colab disconnect
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# Confirm a GPU is available (training on CPU would be very slow)
print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Mounted at /content/drive
TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration
All key settings in one place. `SEED`, `IMG_SIZE` and the 20% validation split must stay the same as the Classic-CNN notebook so both models see the same images.

In [2]:
SEED = 42                 # same seed as Classic-CNN -> same train/val split
IMG_SIZE = (224, 224)     # DenseNet121's native input size
BATCH_SIZE = 32
VAL_SPLIT = 0.2           # 80% train / 20% validation

STAGE1_EPOCHS = 10        # head training (frozen backbone)
STAGE2_EPOCHS = 20        # fine-tuning
STAGE1_LR = 1e-3
STAGE2_LR = 1e-5          # small LR so pre-trained weights are not destroyed
UNFREEZE_LAST = 60        # number of DenseNet121 layers to unfreeze in stage 2

# All outputs (models, plots) are saved here on Google Drive
OUT_DIR = "/content/drive/MyDrive/DL Assignment/Data/DR_DenseNet121_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)   # reproducible weights, shuffling and augmentation

## 3. Load the dataset
The dataset is stored in Google Drive as a zip file (`My Drive/DL Assignment/Data.zip`).
It is unzipped to Colab's local disk, which is much faster than reading thousands of small files from Drive.

In [3]:
# ---- Load data from Google Drive (zip file) ----
# My Drive > DL Assignment > Data.zip
ZIP_PATH = "/content/drive/MyDrive/DL Assignment/Data.zip"

# Unzip to Colab's local disk (fast, and avoids incomplete folder uploads)
DATA_ROOT = "/content/data"
if os.path.isdir(DATA_ROOT):
    shutil.rmtree(DATA_ROOT)          # remove any previous incomplete copy
print("Unzipping dataset...")
shutil.unpack_archive(ZIP_PATH, DATA_ROOT)
print("Done.")

# Find the folder that directly contains the 5 class folders (works however the zip is nested)
DATA_DIR = next(d for d, sub, _ in os.walk(DATA_ROOT) if "No_DR" in sub and "Mild" in sub)
print("DATA_DIR:", DATA_DIR)

# Check image counts per class (must total 3662)
total = 0
for cls in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, cls)
    if os.path.isdir(p):
        n = len(os.listdir(p)); total += n
        print(f"{cls:15s} {n}")
print("Total:", total)
assert total == 3662, "Dataset incomplete"

Unzipping dataset...
Done.
DATA_DIR: /content/data/gaussian_filtered_images/gaussian_filtered_images
Mild            370
Moderate        999
No_DR           1805
Proliferate_DR  295
Severe          193
Total: 3662


## 4. Train / validation split
Uses **exactly the same two calls as the Classic-CNN notebook**, so the split is identical.

⚠️ Do **not** add `class_names=` or `shuffle=False` here — both change the file order before the seeded shuffle and would produce a different split.

Expected output: **2930 files for training, 732 files for validation.**

In [4]:
common = dict(validation_split=VAL_SPLIT, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

raw_train = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset="training", **common)
raw_val   = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset="validation", **common)

print("Folder order (Keras labels):", raw_train.class_names)

Found 3662 files belonging to 5 classes.
Using 2930 files for training.
Found 3662 files belonging to 5 classes.
Using 732 files for validation.
Folder order (Keras labels): ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']


## 5. Label remapping and preprocessing
**Label order:** Keras numbers classes alphabetically (Mild=0, Moderate=1, No_DR=2, ...). APTOS uses clinical severity order (0 = No DR → 4 = Proliferative DR). We remap to the clinical order so that *quadratic weighted kappa* — the official APTOS metric — is meaningful.

**Preprocessing:** DenseNet121's ImageNet weights expect `densenet.preprocess_input` (scale to 0–1, then normalise with ImageNet mean/std), not plain `/255`.

In [5]:
CLASS_ORDER = ["No_DR", "Mild", "Moderate", "Severe", "Proliferate_DR"]   # clinical order 0..4
LABEL_OF = {name: i for i, name in enumerate(CLASS_ORDER)}

# Lookup table: alphabetical Keras label -> clinical label
REMAP = tf.constant([LABEL_OF[name] for name in raw_train.class_names])
preprocess = tf.keras.applications.densenet.preprocess_input

def fix(images, labels):
    return preprocess(images), tf.gather(REMAP, labels)

AUTOTUNE = tf.data.AUTOTUNE
# cache() keeps decoded images in memory after the first epoch -> faster training
train_ds = raw_train.map(fix, num_parallel_calls=AUTOTUNE).cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
# cache() on validation also fixes its order, so predictions and labels always line up
val_ds   = raw_val.map(fix, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

## 6. Class weights
The dataset is heavily imbalanced (No_DR ≈ 1805 images vs Severe ≈ 193). Class weights make mistakes on rare classes cost more, so the model doesn't just predict No_DR.

In [6]:
# Read labels from the file paths (no need to load the images)
train_labels = np.array([LABEL_OF[os.path.basename(os.path.dirname(p))] for p in raw_train.file_paths])
val_labels   = np.array([LABEL_OF[os.path.basename(os.path.dirname(p))] for p in raw_val.file_paths])

print(pd.DataFrame({"train": np.bincount(train_labels, minlength=5),
                    "val":   np.bincount(val_labels, minlength=5)}, index=CLASS_ORDER))

weights = compute_class_weight("balanced", classes=np.arange(5), y=train_labels)
CLASS_WEIGHT = dict(enumerate(weights))
print("\nClass weights:", {CLASS_ORDER[k]: round(v, 2) for k, v in CLASS_WEIGHT.items()})

                train  val
No_DR            1442  363
Mild              311   59
Moderate          794  205
Severe            147   46
Proliferate_DR    236   59

Class weights: {'No_DR': np.float64(0.41), 'Mild': np.float64(1.88), 'Moderate': np.float64(0.74), 'Severe': np.float64(3.99), 'Proliferate_DR': np.float64(2.48)}


## 7. Model architecture
- **Data augmentation** (training only): random flips, small rotations and zoom — fundus images have no fixed orientation.
- **Backbone:** DenseNet121 with ImageNet weights, without its original 1000-class head.
- **New head:** Global Average Pooling → Dropout(0.3) → Dense(5, softmax).

`training=False` keeps the backbone's BatchNorm layers in inference mode, which keeps fine-tuning stable.

In [7]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1),    # up to ±36 degrees
    tf.keras.layers.RandomZoom(0.1),
], name="augmentation")

base = tf.keras.applications.DenseNet121(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
base.trainable = False    # frozen for stage 1

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(5, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="densenet121_dr")
model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "densenet121_dr"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         5,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,042,629 (26.87 MB)

 Trainable params: 5,125 (20.02 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

## 8. Callbacks
- **EarlyStopping** — stops when validation loss stops improving and restores the best weights.
- **ReduceLROnPlateau** — lowers the learning rate when progress stalls.
- **ModelCheckpoint** — saves the best model to Google Drive.

In [8]:
def get_callbacks(stage):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/densenet121_{stage}_best.keras",
                                           monitor="val_loss", save_best_only=True),
    ]

## 9. Stage 1 — Train the classification head
Only the new Dense layer learns; the DenseNet121 backbone is used as a fixed feature extractor.

In [9]:
model.compile(optimizer=tf.keras.optimizers.Adam(STAGE1_LR),
              loss="sparse_categorical_crossentropy",   # labels are integers 0..4
              metrics=["accuracy"])

hist1 = model.fit(train_ds, validation_data=val_ds, epochs=STAGE1_EPOCHS,
                  class_weight=CLASS_WEIGHT, callbacks=get_callbacks("stage1"))

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 50s 276ms/step - accuracy: 0.4420 - loss: 1.5820 - val_accuracy: 0.6612 - val_loss: 0.9984 - learning_rate: 0.0010
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 184ms/step - accuracy: 0.5676 - loss: 1.3465 - val_accuracy: 0.6872 - val_loss: 0.8889 - learning_rate: 0.0010
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 194ms/step - accuracy: 0.6024 - loss: 1.2285 - val_accuracy: 0.6311 - val_loss: 0.9511 - learning_rate: 0.0010
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 189ms/step - accuracy: 0.6348 - loss: 1.2006 - val_accuracy: 0.7268 - val_loss: 0.8279 - learning_rate: 0.0010
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 178ms/step - accuracy: 0.6317 - loss: 1.1593 - val_accuracy: 0.6585 - val_loss: 0.9129 - learning_rate: 0.0010
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 178ms/step - accuracy: 0.6488 - loss: 1.1394 - val_accuracy: 0.6352 - val_loss: 0.9189 - learning_rate: 0.0010
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 178ms/step - accuracy: 0.6611 - loss: 1.